# AgriScan re-implementation: second protocol (grouped splits)

Builds a photo-grouped and a leaf+photo-grouped 70/15/15 split from the dataset folder, then trains MobileNetV2 and EfficientNet-B3
(one seed each) on the leaf-grouped split (primary) and the photo-grouped split. About 1.6 h on a T4 (an estimate).

**Before running** (right sidebar): Accelerator = **GPU T4 x2 (or P100)**, **Internet = On** (the authors' leaf map is downloaded at run time),
and add two inputs: the private code-bundle dataset **at its latest version**, and the public **New Plant Diseases Dataset**.
Then **Save Version -> Save & Run All (Commit)**. Everything is written to `/kaggle/working/out2`; see `KAGGLE_RUN.md`.
The split is rebuilt here and compared with the fingerprints committed in the repo: a mismatch stops the run before any training.
Never paste tokens or passwords here.

In [ ]:
!nvidia-smi
import torch, sys
print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: turn the accelerator on (right sidebar) and restart.'

In [ ]:
import glob, os, shutil, subprocess, zipfile
WORK = '/kaggle/working/agriscan'
GITHUB_URL = None   # alternative to the bundle once the repo is public, e.g. 'https://github.com/<user>/agriscan-reimpl'

def find_code():
    # An input can be mounted several folders deep (e.g. /kaggle/input/datasets/<owner>/<name>/), so look up to 3 levels down.
    for depth in (1, 2, 3):
        stars = '/'.join(['*'] * depth)
        for p in glob.glob(f'/kaggle/input/{stars}/pyproject.toml'):
            if 'name = "agriscan"' in open(p).read():
                return os.path.dirname(p)
        for z in glob.glob(f'/kaggle/input/{stars}/*.zip'):
            if 'agriscan' in os.path.basename(z):
                zipfile.ZipFile(z).extractall('/kaggle/working/_unzipped')
                return '/kaggle/working/_unzipped'
    return None

src = find_code()
if src:
    shutil.copytree(src, WORK, dirs_exist_ok=True)
    print('code copied from', src)
elif GITHUB_URL:
    subprocess.run(['git', 'clone', GITHUB_URL, WORK], check=True)
else:
    raise SystemExit('Code not found: add the private code-bundle dataset as an input (see KAGGLE_RUN.md), or set GITHUB_URL.')
print(open(WORK + '/BUNDLE_INFO.json').read() if os.path.exists(WORK + '/BUNDLE_INFO.json') else 'git clone, no BUNDLE_INFO.json')

In [ ]:
# Kaggle ships torch/torchvision; this adds the rest (ultralytics = YOLO stage, pytest = quick self-test).
!pip install -q -e "/kaggle/working/agriscan[dev,app,detect]"

In [ ]:
# 3-4 minutes on CPU: logic tests on synthetic data. A failure here means do not trust the run below.
!cd /kaggle/working/agriscan && python -m pytest -q -x tests

In [ ]:
# Safe to re-run after a crash: finished stages are skipped.
!cd /kaggle/working/agriscan && python scripts/run_grouped.py --data auto --workdir /kaggle/working/out2 --max-hours 2.5

In [ ]:
# What to download (Output tab): export/agriscan_grouped_results.zip (and export/models/*.pt if you want the leaf-split checkpoint)
!ls -la /kaggle/working/out2/export /kaggle/working/out2/export/models
!cat /kaggle/working/out2/results/run_grouped_status.json